# Paper 2 demo: nonconvex transcript Ball-DP

This notebook is a single-dataset tutorial for the theorem-backed nonconvex transcript API. It shows both sides of Paper 2:

1. utility and Hayes-style transcript bounds for a one-hidden-layer operator-norm network;
2. one exact finite-prior transcript MAP attack using the recorded DP-SGD transcript.

The official scripts perform the large sweeps and aggregation. This notebook is deliberately smaller and emphasizes how the library pieces fit together.


## Transcript model

At DP-SGD step \(t\), the sanitized transcript contains a noisy clipped-gradient aggregate
\[
\widetilde S_t = \sum_{i\in B_t}\operatorname{clip}_C(\nabla_\theta\ell(\theta_t;z_i))+\xi_t,
\qquad \xi_t\sim\mathcal N(0,\nu_t^2 I).
\]
The theorem gives a certified record-gradient Lipschitz constant \(L_z\), so a same-label radius-\(r\) replacement has per-step signal bounded by
\[
\Delta_t^{\mathrm{Ball}}(r)\le \min\{L_zr,2C\}.
\]
The standard replacement comparator uses \(2C\). The finite-prior bound used below is the Hayes product-reference bound exposed as
`ball_rero(release, mode="hayes")`.


In [ ]:
from __future__ import annotations

from types import SimpleNamespace
import sys
from pathlib import Path

import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp import (
    ArrayDataset,
    make_trace_metadata_from_release,
    summarize_embedding_ball_radii,
)
from quantbayes.ball_dp.attacks.gradient_based import subtract_known_batch_gradients
from quantbayes.ball_dp.experiments.run_attack_experiment import (
    DEFAULT_ORDERS,
    LoadedDataset,
    find_feasible_support_banks,
    load_embeddings,
    make_support_set,
    radius_value_from_report,
    remove_train_index,
    resolve_dataset,
)
from quantbayes.ball_dp.experiments.run_nonconvex_transcript_experiment import (
    calibrate_noise,
    compute_hayes_bounds,
    resolve_task,
    stratified_subsample,
    train_one_release,
)
from quantbayes.ball_dp.experiments.run_nonconvex_transcript_attack_experiment import (
    run_one_attack,
    train_recorded_release,
)
from quantbayes.ball_dp.theorem import (
    TheoremBounds,
    TheoremModelSpec,
    certified_lz,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})


## Configuration

The defaults are intentionally modest. Increase `NUM_STEPS`, subset sizes, and seeds in the official scripts rather than in the tutorial.


In [ ]:
DATASET = "MNIST-embeddings"
MAX_TRAIN = 1024
MAX_TEST = 512
SUBSAMPLE_SEED = 0

HIDDEN_DIM = 128
A = 4.0
LAMBDA = 4.0
INPUT_BOUND_MARGIN = 1.001

RADIUS_TAG = "q80"
EPSILON = 8.0
DELTA = 1e-6
M = 8

NUM_STEPS = 100
BATCH_SIZE = 128
CLIP_NORM = 1.0
LEARNING_RATE = 3e-3
EVAL_EVERY = 50
RELEASE_SEED = 0

loader_args = SimpleNamespace(
    data_root="./data",
    embedding_batch_size=256,
    allow_cpu_fallback=True,
    embedding_cache_path=None,
    force_recompute_embeddings=False,
    num_workers=2,
    hf_cache_dir=None,
    encoder_model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=256,
)


## Load embeddings and estimate the policy radius

The metric is the same label-preserving embedding metric used in Paper 1. The transcript theorem then maps this feature-space radius through the certified Lipschitz constant \(L_z\).


In [ ]:
dataset_spec = resolve_dataset(DATASET)
loaded = load_embeddings(loader_args, dataset_spec)
X_train, y_train = stratified_subsample(
    loaded.X_train, loaded.y_train, max_examples=MAX_TRAIN, seed=SUBSAMPLE_SEED
)
X_test, y_test = stratified_subsample(
    loaded.X_test, loaded.y_test, max_examples=MAX_TEST, seed=SUBSAMPLE_SEED + 17
)

num_classes = int(len(np.unique(y_train)))
feature_dim = int(X_train.shape[1])
task = resolve_task("auto", num_classes)

radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95),
    max_exact_pairs=250_000,
    max_sampled_pairs=100_000,
    seed=SUBSAMPLE_SEED,
)
radius = radius_value_from_report(radius_report, RADIUS_TAG)

pd.DataFrame([
    {
        "dataset": DATASET,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "task": task,
        "radius_tag": RADIUS_TAG,
        "radius": radius,
    }
])


## Build the theorem-backed model specification

The public bounds \((B,A,\Lambda)\) define the constrained network class and the certified Lipschitz constant \(L_z\).


In [ ]:
B_all = float(
    max(
        np.max(np.linalg.norm(X_train, axis=1)),
        np.max(np.linalg.norm(X_test, axis=1)),
    ) * INPUT_BOUND_MARGIN
)
bounds = TheoremBounds(B=B_all, A=A, Lambda=LAMBDA)
spec = TheoremModelSpec(
    d_in=feature_dim,
    hidden_dim=HIDDEN_DIM,
    task=task,
    parameterization="dense",
    constraint="op",
    num_classes=(None if task == "binary" else num_classes),
)
lz = certified_lz(spec, bounds)
critical_radius = 2.0 * CLIP_NORM / max(lz, 1e-30)

pd.DataFrame([
    {
        "B": B_all,
        "A": A,
        "Lambda": LAMBDA,
        "hidden_dim": HIDDEN_DIM,
        "L_z": lz,
        "radius": radius,
        "L_z * radius": lz * radius,
        "2C": 2.0 * CLIP_NORM,
        "Ball step sensitivity": min(lz * radius, 2.0 * CLIP_NORM),
        "critical radius 2C/L_z": critical_radius,
    }
])


## Calibrate DP-SGD noise and train releases

Ball calibration uses \(\min\{L_zr,2C\}\); standard calibration uses \(2C\). Both releases then use the same optimizer and architecture.


In [ ]:
def calibrate(mechanism: str):
    return calibrate_noise(
        target_epsilon=EPSILON,
        mechanism=mechanism,
        n_total=len(X_train),
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        clip_norm=CLIP_NORM,
        radius=radius,
        lz=lz,
        delta=DELTA,
        orders=DEFAULT_ORDERS,
        calibration_steps=18,
    )

cal_ball = calibrate("ball")
cal_standard = calibrate("standard")

release_ball = train_one_release(
    spec=spec,
    bounds=bounds,
    X_train=X_train,
    y_train=y_train,
    X_eval=X_test,
    y_eval=y_test,
    mechanism="ball",
    radius=radius,
    epsilon=EPSILON,
    delta=DELTA,
    noise_multiplier=float(cal_ball["noise_multiplier"]),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    eval_batch_size=512,
    checkpoint_selection="last",
    release_seed=RELEASE_SEED,
    orders=DEFAULT_ORDERS,
    record_operator_norms=False,
    operator_norms_every=100,
)
release_standard = train_one_release(
    spec=spec,
    bounds=bounds,
    X_train=X_train,
    y_train=y_train,
    X_eval=X_test,
    y_eval=y_test,
    mechanism="standard",
    radius=radius,
    epsilon=EPSILON,
    delta=DELTA,
    noise_multiplier=float(cal_standard["noise_multiplier"]),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    eval_batch_size=512,
    checkpoint_selection="last",
    release_seed=RELEASE_SEED,
    orders=DEFAULT_ORDERS,
    record_operator_norms=False,
    operator_norms_every=100,
)


In [ ]:
def first_epsilon(release, view: str):
    ledger = release.privacy.ball if view == "ball" else release.privacy.standard
    certs = getattr(ledger, "dp_certificates", [])
    return np.nan if not certs else float(certs[0].epsilon)

rows = []
for mechanism, release, cal in [
    ("ball", release_ball, cal_ball),
    ("standard", release_standard, cal_standard),
]:
    bounds_row = compute_hayes_bounds(
        release,
        m=M,
        feature_dim=feature_dim,
        include_rdp=True,
    )
    rows.append({
        "mechanism": mechanism,
        "accuracy": release.utility_metrics.get("accuracy", np.nan),
        "public_eval_loss": release.utility_metrics.get("public_eval_loss", np.nan),
        "calibrated_noise_multiplier": cal["noise_multiplier"],
        "calibrated_epsilon": cal["epsilon"],
        "epsilon_ball_view": first_epsilon(release, "ball"),
        "epsilon_standard_view": first_epsilon(release, "standard"),
        "Delta_ball_max": release.sensitivity.delta_ball,
        "Delta_std_max": release.sensitivity.delta_std,
        **bounds_row,
    })
summary = pd.DataFrame(rows)
display(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
axes[0].bar(summary["mechanism"], summary["calibrated_noise_multiplier"])
axes[0].set_ylabel("Noise multiplier")
axes[0].set_title("Calibration")

axes[1].bar(summary["mechanism"], summary["bound_hayes_ball"])
axes[1].axhline(1.0 / M, linestyle="--", color="gray", label=f"baseline 1/{M}")
axes[1].set_ylabel("Hayes exact-ID upper bound")
axes[1].set_ylim(0, 1.02)
axes[1].legend()
axes[1].set_title("Transcript bound")
plt.show()


## One recorded transcript attack

We now create one finite support \(S\), replace one anchor by one target candidate, record the sanitized DP-SGD transcript, residualize known non-target batch contributions, and run both known-inclusion (KI) and unknown-inclusion (UI) MAP attacks.


In [ ]:
small_data = LoadedDataset(
    spec=loaded.spec,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    label_values=loaded.label_values,
    num_classes=num_classes,
    feature_dim=feature_dim,
    empirical_embedding_bound=float(np.max(np.linalg.norm(X_train, axis=1))),
    backend=loaded.backend,
)

banks = find_feasible_support_banks(
    small_data,
    radius_value=radius,
    max_required_m=M,
    num_supports=1,
    anchor_seed=0,
    source_mode="public_only",
    max_search=2000,
    explicit_anchor_indices=None,
    strict=True,
    anchor_selection="rare_class",
)
bank = banks[0]
X_support, y_support, source_ids, dists = make_support_set(
    bank,
    m=M,
    draw_index=0,
    base_seed=0,
    selection="farthest",
)

target_pos = 0
x_target = X_support[target_pos]
y_target = int(y_support[target_pos])
X_minus, y_minus = remove_train_index(small_data.X_train, small_data.y_train, bank.anchor_index)
X_full = np.concatenate([X_minus, x_target[None, :]], axis=0).astype(np.float32)
y_full = np.concatenate([y_minus, np.asarray([y_target], dtype=np.int32)], axis=0)
target_index = len(X_full) - 1
full_ds = ArrayDataset(X_full, y_full, name="demo_attack_full")
true_record = full_ds.record(target_index)

display(pd.DataFrame({
    "candidate": np.arange(M),
    "source_id": source_ids,
    "distance_to_anchor": dists,
    "is_target": [i == target_pos for i in range(M)],
}))


In [ ]:
release_attack, recorder = train_recorded_release(
    spec=spec,
    bounds=bounds,
    X_train=X_full,
    y_train=y_full,
    X_eval=X_test,
    y_eval=y_test,
    mechanism="ball",
    radius=radius,
    epsilon=EPSILON,
    delta=DELTA,
    noise_multiplier=float(cal_ball["noise_multiplier"]),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    eval_batch_size=512,
    checkpoint_selection="last",
    release_seed=RELEASE_SEED,
    orders=DEFAULT_ORDERS,
    trace_capture_every=1,
    record_operator_norms=False,
    operator_norms_every=50,
)

trace_meta = make_trace_metadata_from_release(release_attack, target_index=target_index)
trace = recorder.to_trace(
    state=release_attack.extra.get("model_state", None),
    loss_name=spec.loss_name,
    reduction=str(trace_meta.get("reduction", "mean")),
    metadata=trace_meta,
)
target_inclusions = sum(
    bool(np.any(np.asarray(step.batch_indices, dtype=np.int64) == target_index))
    for step in trace.steps
)
print({"retained_steps": len(trace.steps), "target_inclusions": target_inclusions})

residual_trace = subtract_known_batch_gradients(
    trace,
    full_ds,
    target_index=target_index,
    loss_name=spec.loss_name,
    seed=0,
)


In [ ]:
attack_rows = []
for mode in ["known_inclusion", "unknown_inclusion"]:
    status, metrics, diagnostics = run_one_attack(
        residual_trace=residual_trace,
        X_support=X_support,
        y_support=y_support,
        target_index=target_index,
        known_label=int(bank.anchor_label),
        true_record=true_record,
        mode=mode,
        attack_seed=0,
        known_inclusion_step_mode="present_steps",
    )
    attack_rows.append({
        "mode": mode,
        "status": status,
        "exact_identification_success": metrics.get("exact_identification_success", np.nan),
        "prior_rank": metrics.get("prior_rank", np.nan),
        "predicted_prior_index": diagnostics.get("predicted_prior_index"),
        "true_prior_index": diagnostics.get("true_prior_index"),
        "selected_step_count": diagnostics.get("selected_step_count"),
    })
attack_df = pd.DataFrame(attack_rows)
display(attack_df)


## Relation to the official scripts

The official Paper 2 scripts repeat this workflow across datasets, seeds, privacy levels, supports, and attack modes. This notebook demonstrates the same objects and APIs on one compact run.
